# AI Document Extraction Colab

Upload a document, extract invoice fields, validate them, review corrections, and save outputs.

In [ ]:
!pip -q install pandas pypdf pdfplumber unstructured ipywidgets

In [ ]:
from __future__ import annotations

import base64, json, re, sqlite3
from dataclasses import dataclass, asdict, field
from datetime import datetime
from difflib import SequenceMatcher
from pathlib import Path
from typing import Any

import pandas as pd
import ipywidgets as widgets
from IPython.display import HTML, Markdown, display
from google.colab import files

DATA_DIR = Path('colab_data')
UPLOAD_DIR = DATA_DIR / 'uploads'
OUTPUT_DIR = DATA_DIR / 'outputs'
DB_PATH = DATA_DIR / 'document_results.db'
TEMPLATE_PATH = DATA_DIR / 'learned_templates.json'
for d in (DATA_DIR, UPLOAD_DIR, OUTPUT_DIR): d.mkdir(parents=True, exist_ok=True)
if not TEMPLATE_PATH.exists(): TEMPLATE_PATH.write_text('[]', encoding='utf-8')
FIELD_ORDER = ['document_type','vendor_name','invoice_number','invoice_date','due_date','subtotal','tax','total_amount','currency']

@dataclass
class ValidationCheck:
    field: str
    status: str
    message: str
    def to_dict(self): return asdict(self)

@dataclass
class ParsedDocument:
    file_name: str
    file_path: Path
    raw_text: str
    sections: list[str] = field(default_factory=list)

@dataclass
class PipelineResult:
    source_file: str
    upload_path: str
    parsed_text: str
    extracted_data: dict[str, Any]
    validation_results: list[dict[str, str]]
    output_files: dict[str, str]
    summary: dict[str, Any]
    errors: list[str]
    extraction_trace: list[str] = field(default_factory=list)
    processed_at: str = field(default_factory=lambda: datetime.utcnow().isoformat() + 'Z')

class TemplateMemory:
    def __init__(self, path: Path): self.path = path
    def load(self):
        try: return json.loads(self.path.read_text(encoding='utf-8'))
        except json.JSONDecodeError: return []
    def best_match(self, signature):
        best = None
        for template in self.load():
            score = self.score(signature, template.get('signature', {}))
            if best is None or score > best['score']: best = {'template': template, 'score': score}
        return best
    def learn(self, source_file, signature, extracted_data, lines):
        templates = self.load()
        anchors = {}
        for field, value in extracted_data.items():
            if field in {'document_type','source_file'} or value in (None,'',[]): continue
            for line in lines:
                if str(value).lower() in line.lower():
                    label = line.split(':', 1)[0].strip() if ':' in line else ''
                    anchors[field] = {'pattern': rf"{re.escape(label)}\s*:\s*(?P<value>.+)" if label else re.escape(str(value))}
                    break
        template = {'template_name': Path(source_file).stem, 'signature': signature, 'anchors': anchors}
        match = self.best_match(signature)
        if match and match['score'] >= 0.92:
            template['template_name'] = match['template'].get('template_name', template['template_name'])
            templates = [template if t.get('template_name') == template['template_name'] else t for t in templates]
        else:
            templates.append(template)
        self.path.write_text(json.dumps(templates, indent=2), encoding='utf-8')
        return template
    @staticmethod
    def signature(lines):
        top = [re.sub(r'\s+', ' ', re.sub(r'\d', '#', line.lower())).strip() for line in lines[:8]]
        joined = ' '.join(lines[:20]).lower()
        keywords = sorted({k for k in ('invoice','vendor','supplier','total','tax','due date') if k in joined})
        return {'top_lines': top, 'keywords': keywords}
    @staticmethod
    def score(left, right):
        left_text = ' | '.join(left.get('top_lines', []))
        right_text = ' | '.join(right.get('top_lines', []))
        a = SequenceMatcher(None, left_text, right_text).ratio()
        lk, rk = set(left.get('keywords', [])), set(right.get('keywords', []))
        b = 1.0 if not lk and not rk else len(lk & rk) / max(len(lk | rk), 1)
        return round((a * 0.7) + (b * 0.3), 4)

class DocumentParser:
    def parse(self, file_path: Path):
        suffix = file_path.suffix.lower()
        if suffix in {'.txt', '.md'}:
            raw = file_path.read_text(encoding='utf-8', errors='ignore')
        elif suffix == '.json':
            raw = json.dumps(json.loads(file_path.read_text(encoding='utf-8')), indent=2)
        elif suffix == '.pdf':
            raw = self._parse_pdf(file_path)
        else:
            raw = file_path.read_text(encoding='utf-8', errors='ignore')
        sections = [line.strip() for line in raw.splitlines() if line.strip()]
        return ParsedDocument(file_name=file_path.name, file_path=file_path, raw_text=raw, sections=sections)
    def _parse_pdf(self, file_path: Path):
        for parser in (self._with_unstructured, self._with_pypdf, self._with_pdfplumber):
            text = parser(file_path)
            if text: return text
        raise RuntimeError('No extractable PDF text found.')
    def _with_unstructured(self, file_path):
        try:
            from unstructured.partition.pdf import partition_pdf
            return '\n'.join(str(x).strip() for x in partition_pdf(filename=str(file_path)) if str(x).strip())
        except Exception:
            return ''
    def _with_pypdf(self, file_path):
        try:
            from pypdf import PdfReader
            reader = PdfReader(str(file_path))
            return '\n\n'.join((page.extract_text() or '').strip() for page in reader.pages if (page.extract_text() or '').strip())
        except Exception:
            return ''
    def _with_pdfplumber(self, file_path):
        try:
            import pdfplumber
            with pdfplumber.open(str(file_path)) as pdf:
                return '\n\n'.join((page.extract_text() or '').strip() for page in pdf.pages if (page.extract_text() or '').strip())
        except Exception:
            return ''

class InvoiceValidator:
    def validate(self, data):
        checks = []
        for field in ('vendor_name','invoice_number','invoice_date','total_amount'):
            checks.append(ValidationCheck(field, 'pass' if data.get(field) not in (None,'',[]) else 'fail', 'Required field is present.' if data.get(field) not in (None,'',[]) else 'Required field is missing.'))
        for field in ('invoice_date','due_date'):
            value = data.get(field)
            if not value: checks.append(ValidationCheck(field, 'warn', 'No date provided.'))
            elif any(_valid_date(str(value), fmt) for fmt in ('%Y-%m-%d','%m/%d/%Y','%Y/%m/%d')): checks.append(ValidationCheck(field, 'pass', 'Date format is valid.'))
            else: checks.append(ValidationCheck(field, 'fail', 'Date should be in YYYY-MM-DD or MM/DD/YYYY.'))
        total = data.get('total_amount')
        checks.append(ValidationCheck('total_amount', 'pass' if isinstance(total, (int,float)) and total > 0 else 'fail', 'Total amount is positive.' if isinstance(total, (int,float)) and total > 0 else 'Total amount must be > 0.'))
        return checks

def _valid_date(value, fmt):
    try:
        datetime.strptime(value, fmt)
        return True
    except ValueError:
        return False

def empty_invoice(source_file):
    return {'document_type':'invoice','source_file':source_file,'vendor_name':None,'invoice_number':None,'invoice_date':None,'due_date':None,'subtotal':None,'tax':None,'total_amount':None,'currency':'USD'}

def apply_template(template, raw_text):
    out = {}
    for field, anchor in template.get('anchors', {}).items():
        match = re.search(anchor.get('pattern', ''), raw_text, re.IGNORECASE)
        if match:
            value = match.groupdict().get('value') or match.group(0)
            out[field] = value.strip()
    for field in ('subtotal','tax','total_amount'):
        if out.get(field) not in (None, ''):
            try: out[field] = float(str(out[field]).replace(',', '').replace('$', ''))
            except ValueError: pass
    return out

def rule_extract(parsed_document):
    text = parsed_document.raw_text
    data = empty_invoice(parsed_document.file_name)
    patterns = {
        'vendor_name':[r'Vendor\s*:\s*(?P<value>.+)', r'Supplier\s*:\s*(?P<value>.+)', r'From\s*:\s*(?P<value>.+)'],
        'invoice_number':[r'Invoice\s*(?:Number|No\.?)\s*:\s*(?P<value>[\w\-]+)', r'Invoice\s*#\s*(?P<value>[\w\-]+)'],
        'invoice_date':[r'Invoice\s*Date\s*:\s*(?P<value>[\d/\-]+)', r'Date\s*:\s*(?P<value>[\d/\-]+)'],
        'due_date':[r'Due\s*Date\s*:\s*(?P<value>[\d/\-]+)'],
        'subtotal':[r'Subtotal\s*:\s*\$?(?P<value>[\d,]+(?:\.\d{1,2})?)'],
        'tax':[r'Tax\s*:\s*\$?(?P<value>[\d,]+(?:\.\d{1,2})?)'],
        'total_amount':[r'Total\s*:\s*\$?(?P<value>[\d,]+(?:\.\d{1,2})?)', r'Amount\s*Due\s*:\s*\$?(?P<value>[\d,]+(?:\.\d{1,2})?)'],
        'currency':[r'Currency\s*:\s*(?P<value>[A-Z]{3})'],
    }
    for field, plist in patterns.items():
        for pattern in plist:
            m = re.search(pattern, text, re.IGNORECASE)
            if m:
                data[field] = m.group('value').strip()
                break
    for field in ('subtotal','tax','total_amount'):
        if data.get(field) is not None:
            data[field] = float(str(data[field]).replace(',', ''))
    return data

def infer_missing(raw_text, data):
    out = {}
    if data.get('currency') in (None, '') and '$' in raw_text: out['currency'] = 'USD'
    if data.get('invoice_number') in (None, ''):
        m = re.search(r'\b(?:INV|Invoice)[-\s#:]*(\w+)\b', raw_text, re.IGNORECASE)
        if m: out['invoice_number'] = m.group(1)
    if data.get('invoice_date') in (None, ''):
        m = re.search(r'\b(\d{4}-\d{2}-\d{2}|\d{2}/\d{2}/\d{4})\b', raw_text)
        if m: out['invoice_date'] = m.group(1)
    return out

class DocumentPipeline:
    def __init__(self):
        self.parser = DocumentParser()
        self.validator = InvoiceValidator()
        self.templates = TemplateMemory(TEMPLATE_PATH)
    def process_upload(self, file_name, file_bytes):
        path = UPLOAD_DIR / Path(file_name).name
        path.write_bytes(file_bytes)
        parsed = self.parser.parse(path)
        trace = ['Generated document signature from top lines and keywords.']
        signature = TemplateMemory.signature(parsed.sections)
        match = self.templates.best_match(signature)
        if match and match['score'] >= 0.55:
            trace.append(f"Matched template `{match['template'].get('template_name', 'unknown')}` with similarity {match['score']}.")
            extracted = empty_invoice(parsed.file_name)
            extracted.update(apply_template(match['template'], parsed.raw_text))
        else:
            trace.append('No close learned template was found. Falling back to rule-based extraction.')
            extracted = rule_extract(parsed)
        inferred = infer_missing(parsed.raw_text, extracted)
        extracted.update(inferred)
        if inferred: trace.append(f"Inferred extra fields: {', '.join(sorted(inferred))}.")
        checks = self.validator.validate(extracted)
        total, passed = len(checks), sum(check.status == 'pass' for check in checks)
        learned = None
        if total and passed / total >= 0.6:
            learned = self.templates.learn(path.name, signature, extracted, parsed.sections).get('template_name')
            trace.append(f"Learned or updated template `{learned}` from this upload.")
        outputs = self.persist(path.name, extracted, checks, trace)
        return PipelineResult(path.name, str(path), parsed.raw_text, extracted, [c.to_dict() for c in checks], outputs, {'source_file': path.name, 'validation_passes': passed, 'validation_fails': sum(c.status == 'fail' for c in checks), 'validation_warnings': sum(c.status == 'warn' for c in checks), 'learned_template': learned}, [], trace)
    def finalize_review(self, result, corrected_data):
        trace = ['Used human-reviewed corrections from the notebook UI.']
        checks = self.validator.validate(corrected_data)
        signature = TemplateMemory.signature([line.strip() for line in result.parsed_text.splitlines() if line.strip()])
        total, passed = len(checks), sum(c.status == 'pass' for c in checks)
        learned = None
        if total and passed / total >= 0.6:
            learned = self.templates.learn(result.source_file, signature, corrected_data, [line.strip() for line in result.parsed_text.splitlines() if line.strip()]).get('template_name')
            trace.append(f"Learned or updated template `{learned}` from reviewed data.")
        outputs = self.persist(result.source_file, corrected_data, checks, trace)
        return PipelineResult(result.source_file, result.upload_path, result.parsed_text, corrected_data, [c.to_dict() for c in checks], outputs, {'source_file': result.source_file, 'reviewed_by_user': True, 'learned_template': learned}, [], trace)
    def persist(self, source_file, extracted, checks, trace):
        stem = Path(source_file).stem
        json_path, csv_path, trace_path = OUTPUT_DIR / f'{stem}.json', OUTPUT_DIR / f'{stem}.csv', OUTPUT_DIR / f'{stem}_trace.json'
        json_path.write_text(json.dumps(extracted, indent=2), encoding='utf-8')
        trace_path.write_text(json.dumps({'trace': trace}, indent=2), encoding='utf-8')
        pd.DataFrame([extracted]).to_csv(csv_path, index=False)
        conn = sqlite3.connect(DB_PATH)
        try:
            pd.DataFrame([{'source_file': source_file, **extracted}]).to_sql('document_results', conn, if_exists='append', index=False)
            pd.DataFrame([{'source_file': source_file, **c.to_dict()} for c in checks]).to_sql('validation_results', conn, if_exists='append', index=False)
        finally:
            conn.close()
        return {'json': str(json_path), 'csv': str(csv_path), 'trace': str(trace_path), 'sqlite': str(DB_PATH)}

pipeline = DocumentPipeline()
print('Colab pipeline ready.')

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload a PDF, TXT, MD, or JSON file first.')
file_name, file_bytes = next(iter(uploaded.items()))
result = pipeline.process_upload(file_name, file_bytes)
display(Markdown('## Summary'))
display(pd.DataFrame([result.summary]))
display(Markdown('## Extracted fields'))
display(pd.DataFrame([result.extracted_data]))
display(Markdown('## Validation report'))
display(pd.DataFrame(result.validation_results))
display(Markdown('## Agent trace'))
for step in result.extraction_trace: print('-', step)

In [ ]:
display(Markdown('## Source preview'))
path = Path(result.upload_path)
if path.suffix.lower() == '.pdf' and path.exists():
    encoded = base64.b64encode(path.read_bytes()).decode('utf-8')
    display(HTML(f'<iframe src="data:application/pdf;base64,{encoded}" width="100%" height="700"></iframe>'))
else:
    print('Inline PDF preview is only available for PDF uploads.')
display(Markdown('## Copyable parsed text'))
print(result.parsed_text[:20000])

In [ ]:
widgets_map = {}
for field in FIELD_ORDER:
    widgets_map[field] = widgets.Text(value='' if result.extracted_data.get(field) is None else str(result.extracted_data.get(field)), description=field + ':', layout=widgets.Layout(width='95%'), style={'description_width': '140px'})
button = widgets.Button(description='Save reviewed result', button_style='success')
out = widgets.Output()

def coerce(values):
    corrected = {'source_file': result.source_file}
    for field, value in values.items():
        text = value.strip()
        if field in {'subtotal','tax','total_amount'}:
            if not text: corrected[field] = None
            else:
                try: corrected[field] = float(text.replace(',', '').replace('$', ''))
                except ValueError: corrected[field] = text
        else:
            corrected[field] = text or None
    corrected['document_type'] = corrected.get('document_type') or 'invoice'
    corrected['currency'] = corrected.get('currency') or 'USD'
    return corrected

def save_review(_):
    corrected = coerce({field: widget.value for field, widget in widgets_map.items()})
    reviewed = pipeline.finalize_review(result, corrected)
    with out:
        out.clear_output()
        display(Markdown('## Reviewed summary'))
        display(pd.DataFrame([reviewed.summary]))
        display(Markdown('## Reviewed extracted fields'))
        display(pd.DataFrame([reviewed.extracted_data]))
        display(Markdown('## Reviewed validation report'))
        display(pd.DataFrame(reviewed.validation_results))

button.on_click(save_review)
display(Markdown('## Review and correct fields'))
display(widgets.VBox(list(widgets_map.values()) + [button, out]))